In [4]:
import yaml

def get_config():
    with open('config.yaml', encoding='utf-8') as f:
        config = yaml.safe_load(f)
        return config

config = get_config()

ModuleNotFoundError: No module named 'yaml'

In [ ]:
%pip install h5py

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import h5py
import numpy as np
from pathlib import Path

In [ ]:
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import matplotlib.pyplot as plt

In [13]:
root = '/home/hyunjin/RBY1_migration/rby1_ws/rby1-data-collection/Demo/'
task_name = config['conversion_task_name']
demo_root = os.path.join(root, task_name)

RAW_H5_DIR = Path(demo_root)
# print(RAW_H5_DIR / "demo_103.h5")
# h5_files = sorted(list(RAW_H5_DIR.glob("*.h5")))
# print(h5_files)
# h5_file_path = h5_files[-1]
h5_file_path = RAW_H5_DIR / "demo_178.h5"

NameError: name 'config' is not defined

In [ ]:
# Create a tree visualization
def create_tree_visualization(file_path):
    """Create a tree-like visualization of the HDF5 structure"""
    
    def tree_structure(name, obj, prefix="", is_last=True):
        """Recursively create tree structure"""
        connector = "└── " if is_last else "├── "
        
        if isinstance(obj, h5py.Group):
            print(f"{prefix}{connector}📁 {name.split('/')[-1] if '/' in name else name}")
        elif isinstance(obj, h5py.Dataset):
            shape_str = f"{obj.shape}" if len(obj.shape) > 0 else "scalar"
            print(f"{prefix}{connector}📄 {name.split('/')[-1] if '/' in name else name} {shape_str} [{obj.dtype}]")
    
    with h5py.File(file_path, 'r') as f:
        print("\n" + "="*70)
        print("Tree Structure Visualization:")
        print("="*70)
        print(f"\n📦 {file_path.name}")
        
        items = list(f.items())
        for i, (name, obj) in enumerate(items):
            is_last = (i == len(items) - 1)
            tree_structure(name, obj, "", is_last)
            
            # If it's a group, show its children
            if isinstance(obj, h5py.Group):
                sub_items = list(obj.items())
                for j, (sub_name, sub_obj) in enumerate(sub_items):
                    sub_is_last = (j == len(sub_items) - 1)
                    sub_prefix = "    " if is_last else "│   "
                    tree_structure(sub_name, sub_obj, sub_prefix, sub_is_last)

create_tree_visualization(h5_file_path)


Tree Structure Visualization:

📦 demo_138.h5
├── 📁 head_depth
├── 📁 head_rgb
├── 📁 pointclouds
└── 📁 samples


In [ ]:
with h5py.File(h5_file_path, 'r') as f:
    # if 'head_rgb' in f and 'image' in f['head_rgb']:
    print(f['head_rgb'])

<HDF5 group "/head_rgb" (0 members)>


In [ ]:
with h5py.File(h5_file_path, 'r') as f:
    # if 'head_rgb' in f and 'image' in f['head_rgb']:
    if 'head_rgb' in f:
        if 'image' in f['head_rgb']:
            print("Image available")
        else:
            print("Image unavailable")

In [ ]:
# Display RGB and Depth images
# h5_file_path = Path('/media/nvidia/T7/Demo/demo_5.h5')

with h5py.File(h5_file_path, 'r') as f:
    if 'head_rgb' in f and 'image' in f['head_rgb']:
        rgb_images = f['head_rgb/image'][:]
        rgb_times = f['head_rgb/time'][:]
        print(f"✅ RGB Images: {rgb_images.shape}")
        print(f"   Frame count: {len(rgb_images)}")
        print(f"   Time range: {rgb_times[0]:.2f}s to {rgb_times[-1]:.2f}s" if len(rgb_times) > 0 else "   No timestamps")
    else:
        rgb_images = None
        print("❌ No RGB images found")
    
    if 'head_depth' in f and 'image' in f['head_depth']:
        depth_images = f['head_depth/image'][:]
            
        depth_times = f['head_depth/time'][:]
        print(f"\n✅ Depth Images: {depth_images.shape}")
        print(f"   Frame count: {len(depth_images)}")
        print(f"   Time range: {depth_times[0]:.2f}s to {depth_times[-1]:.2f}s" if len(depth_times) > 0 else "   No timestamps")
        print(f"   Depth range: {depth_images.min()} to {depth_images.max()} mm")
    else:
        depth_images = None
        print("\n❌ No depth images found")
    
    # Display a sample frame
    if rgb_images is not None and len(rgb_images) > 0:
        sample_idx = len(rgb_images) // 2  # middle frame
        
        fig, axes = plt.subplots(1, 2 if depth_images is not None else 1, figsize=(12, 6))
        
        if depth_images is not None:
            # RGB
            axes[0].imshow(rgb_images[sample_idx])
            axes[0].set_title(f'RGB Image (Frame {sample_idx})')
            axes[0].axis('off')
            
            # Depth (colorized)
            depth_colorized = depth_images[sample_idx].astype(float)
            # Normalize for visualization
            depth_colorized[depth_colorized == 0] = np.nan  # Remove zeros
            im = axes[1].imshow(depth_colorized, cmap='jet')
            axes[1].set_title(f'Depth Image (Frame {sample_idx})')
            axes[1].axis('off')
            plt.colorbar(im, ax=axes[1], label='Depth (mm)')
        else:
            axes.imshow(rgb_images[sample_idx])
            axes.set_title(f'RGB Image (Frame {sample_idx})')
            axes.axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("\nNo images to display")

    # Display all frame
    if rgb_images is not None and len(rgb_images) > 0:
        for sample_idx in range(110,136,3):
            fig, axes = plt.subplots(1, 2 if depth_images is not None else 1, figsize=(12, 6))
            
            if depth_images is not None:
                # RGB
                axes[0].imshow(rgb_images[sample_idx])
                axes[0].set_title(f'RGB Image (Frame {sample_idx})')
                axes[0].axis('off')
                
                # Depth (colorized)
                depth_colorized = depth_images[sample_idx].astype(float)
                # Normalize for visualization
                depth_colorized[depth_colorized == 0] = np.nan  # Remove zeros
                im = axes[1].imshow(depth_colorized, cmap='jet')
                axes[1].set_title(f'Depth Image (Frame {sample_idx})')
                axes[1].axis('off')
                plt.colorbar(im, ax=axes[1], label='Depth (mm)')
            else:
                axes.imshow(rgb_images[sample_idx])
                axes.set_title(f'RGB Image (Frame {sample_idx})')
                axes.axis('off')
            
            plt.tight_layout()
            plt.show()
        else:
            print("\nNo images to display")

❌ No RGB images found

❌ No depth images found

No images to display


In [12]:
# Load and compare robot position vs target joints
with h5py.File(h5_file_path, 'r') as f:
    if 'samples' in f and len(f['samples'].keys()) > 0:
        print("="*70)
        print("Robot Joint Data Analysis")
        print("="*70)
        
        if 'robot_position' in f['samples']:
            robot_pos = f['samples/robot_position'][:]
            print(f"\n✅ Robot Position (Current): {robot_pos.shape}")
            print(f"   First sample: {robot_pos[0] if len(robot_pos) > 0 else 'No data'}")
        
        if 'robot_target_cartesian' in f['samples']:
            target_cart = f['samples/robot_target_cartesian'][:]
            print(f"\n✅ Robot Target (Cartesian): {target_cart.shape}")
            print(f"   Format: [right_xyz(3), left_xyz(3), torso_xyz(3)]")
            print(f"   First sample: {target_cart[0] if len(target_cart) > 0 else 'No data'}")
            for i in range(len(target_cart)):
                print(f"   Sample {i}: {target_cart[i]}")  
        
        if 'robot_target_joints' in f['samples']:
            target_joints = f['samples/robot_target_joints'][:]
            print(f"\n✅ Robot Target (Joint Angles): {target_joints.shape}")
            print(f"   First sample: {target_joints[0] if len(target_joints) > 0 else 'No data'}")
            for i in range(len(target_joints)):
                print(f"   Sample {i}: {target_joints[i]}")
            
            # Check if IK succeeded (not all NaN)
            if len(target_joints) > 0:
                valid_samples = ~np.isnan(target_joints).all(axis=1)
                print(f"   Valid IK solutions: {valid_samples.sum()} / {len(target_joints)}")
                
                if valid_samples.sum() > 0 and 'robot_position' in f['samples']:
                    # Compute difference between target and actual
                    diff = target_joints[valid_samples] - robot_pos[valid_samples]
                    print(f"\n📊 Joint Angle Errors (target - actual):")
                    print(f"   Mean error: {np.rad2deg(np.nanmean(np.abs(diff), axis=0))} degrees")
                    print(f"   Max error:  {np.rad2deg(np.nanmax(np.abs(diff), axis=0))} degrees")
    else:
        print("No sample data found in the file.")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '../Demo/PuttingCupintotheDish/demo_178.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [2]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [15]:
import h5py
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm  # 진행률 표시용 (없으면 pip install tqdm)

def create_video_from_h5(h5_file_path, output_filename='output_video.mp4', fps=30):
    h5_path = Path(h5_file_path)
    
    if not h5_path.exists():
        print(f"❌ 파일을 찾을 수 없습니다: {h5_path}")
        return

    print(f"📂 H5 파일 열기: {h5_path}")

    with h5py.File(h5_path, 'r') as f:
        # 1. 데이터 로드 확인
        # (사용자님 데이터 구조에 맞춰 키 이름 조정: head_rgb, head_depth 등)
        # 만약 키 이름이 다르면 여기서 수정해주세요! (예: 'cam_0/rgb_image' 등)
        rgb_key = 'head_rgb/image' if 'head_rgb/image' in f else None
        depth_key = 'head_depth/image' if 'head_depth/image' in f else None

        # 실제 키가 없는 경우를 대비한 검색 (혹시 이름이 다를까봐)
        if rgb_key is None:
            # 파일 내의 첫 번째 rgb 키를 찾음 (fallback)
            for k in f.keys():
                if 'rgb' in k and 'image' in f[k]:
                    rgb_key = f"{k}/image"
                    break
        
        if depth_key is None:
            for k in f.keys():
                if 'depth' in k and 'image' in f[k]:
                    depth_key = f"{k}/image"
                    break

        if not rgb_key:
            print("❌ RGB 이미지를 찾을 수 없습니다.")
            return

        # 데이터 가져오기
        rgb_images = f[rgb_key][:]
        num_frames, height, width, _ = rgb_images.shape
        print(f"✅ RGB 데이터 로드됨: {num_frames} 프레임 ({width}x{height})")

        has_depth = False
        if depth_key:
            depth_images = f[depth_key][:]
            has_depth = True
            print(f"✅ Depth 데이터 로드됨: {len(depth_images)} 프레임")
        else:
            print("ℹ️ Depth 데이터가 없어 RGB만 저장합니다.")

        # 2. 비디오 설정 (OpenCV)
        # Depth가 있으면 옆으로 붙여서(Side-by-side) 너비를 2배로 설정
        video_width = width * 2 if has_depth else width
        video_height = height
        
        # 코덱 설정 (mp4v는 대부분의 OS에서 호환됨)
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_filename, fourcc, fps, (video_width, video_height))

        print(f"🎬 동영상 생성 시작: {output_filename} ...")

        # 3. 프레임 반복 처리
        for i in range(num_frames):
            # [RGB 처리]
            # OpenCV는 색상 순서가 BGR이므로 RGB -> BGR 변환 필요
            frame_rgb = rgb_images[i]
            frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

            final_frame = frame_bgr

            # [Depth 처리] (있을 경우만)
            if has_depth:
                frame_depth = depth_images[i]
                
                # Depth는 보통 16bit(mm 단위)이므로 화면에 보이게 8bit(0~255)로 정규화해야 함
                # 보기 좋게 만들기: 0~최대거리 대신, 현재 프레임의 min/max 기준으로 정규화
                # (또는 고정된 값 예: 3000mm으로 나눠도 됨)
                depth_norm = cv2.normalize(frame_depth, None, 0, 255, cv2.NORM_MINMAX)
                depth_uint8 = depth_norm.astype(np.uint8)
                
                # 컬러맵 적용 (JET: 파랑(가까움) -> 빨강(멈))
                depth_colormap = cv2.applyColorMap(depth_uint8, cv2.COLORMAP_JET)
                
                # 두 이미지를 가로로 합치기 (Horizontal Stack)
                final_frame = np.hstack((frame_bgr, depth_colormap))

            # 비디오에 쓰기
            out.write(final_frame)
            
            # (선택) 진행 상황 10%마다 출력
            if i % (num_frames // 10) == 0:
                print(f"   Writing frame {i}/{num_frames}")

        # 4. 마무리
        out.release()
        print(f"✨ 동영상 저장 완료! : {output_filename}")


# === 실행 ===
# 여기에 변환하고 싶은 h5 파일 경로를 적어주세요
target_h5_path = h5_file_path = Path('../Demo/PuttingCupintotheDish/demo_170.h5')

create_video_from_h5(target_h5_path, output_filename="demo_video.mp4", fps=30)

❌ 파일을 찾을 수 없습니다: ../Demo/PuttingCupintotheDish/demo_170.h5
